In [1]:
import pandas as pd

In [19]:
import requests
from bs4 import BeautifulSoup

# Pobieranie HTML strony przez requests
url = 'https://adresowo.pl/mieszkania/warszawa/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'Accept-Language': 'pl-PL,pl;q=0.9'
}

response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()
html_content = response.text

soup = BeautifulSoup(html_content, 'html.parser')

# KROK 1: Znajdujemy wszystkie kafelki z ogłoszeniami
offers = soup.find_all('div', attrs={'data-offer-card': True})

scraped_data = []

for offer in offers:
    try:
        a_tag = offer.find('h2').find('a')
        link = 'https://adresowo.pl' + a_tag['href']

        title_spans = a_tag.find_all('span', class_='line-clamp-1')
        title = ' - '.join([span.text.strip() for span in title_spans if span.text.strip()])

        params_container = offer.find('div', class_='flex w-full items-center text-sm')
        params = params_container.find_all('p') if params_container else []

        price = params[0].text.strip().replace('\xa0', ' ') if len(params) > 0 else None
        area = params[1].text.strip() if len(params) > 1 else None
        rooms = params[2].text.strip() if len(params) > 2 else None

        scraped_data.append({
            'Tytuł': title,
            'Cena': price,
            'Metraż': area,
            'Pokoje': rooms,
            'Link': link
        })

    except Exception as e:
        print(f'Pominięto ogłoszenie ze względu na nietypową strukturę: {e}')
        continue

# KROK 4: Tworzymy Pandas DataFrame i sprawdzamy wyniki
df = pd.DataFrame(scraped_data)
print(f'Zescrapowano {len(df)} ofert.')
print(df.head())

Zescrapowano 40 ofert.
                                       Tytuł          Cena  Metraż  Pokoje  \
0                 Warszawa Wola - ul. Pereca    799 000 zł   48 m²  3 pok.   
1             Warszawa Wawer - ul. Goździków  1 010 000 zł   63 m²  3 pok.   
2        Warszawa Mokotów - Limanowskiego 11  1 420 000 zł   99 m²  5 pok.   
3          Warszawa Bemowo - ul. Szeligowska       zapytaj   46 m²  2 pok.   
4  Warszawa Włochy - ul. Aleje Jerozolimskie  2 125 000 zł  125 m²  5 pok.   

                                                Link  
0  https://adresowo.pl/o/mieszkanie-warszawa-wola...  
1  https://adresowo.pl/o/mieszkanie-warszawa-wawe...  
2  https://adresowo.pl/o/mieszkanie-warszawa-moko...  
3  https://adresowo.pl/o/mieszkanie-warszawa-bemo...  
4  https://adresowo.pl/o/mieszkanie-warszawa-wloc...  


In [26]:
df = df[df['Tytuł'].str.contains('Mokotów|Ochota|Rembertów', case=False, na=False)]
df

,Tytuł,Cena,Metraż,Pokoje,Link
2,Warszawa Mokotów - Limanowskiego 11,1 420 000 zł,99 m²,5 pok.,https://adresowo.pl/o/mieszkanie-warszawa-moko...
5,Warszawa Mokotów - ul. Wita Stwosza,851 400 zł,38 m²,1 pok.,https://adresowo.pl/o/mieszkanie-warszawa-moko...
19,Warszawa Mokotów - al. gen. Władysława Sikorsk...,1 790 000 zł,67 m²,3 pok.,https://adresowo.pl/o/mieszkanie-warszawa-moko...
20,Warszawa Mokotów - ul. Jana Maklakiewicza,759 000 zł,48 m²,3 pok.,https://adresowo.pl/o/mieszkanie-warszawa-moko...
21,Warszawa Ochota - ul. Bitwy Warszawskiej 1920 r.,850 000 zł,34 m²,2 pok.,https://adresowo.pl/o/mieszkanie-warszawa-ocho...
22,Warszawa - Rembertów,1 649 000 zł,89 m²,4 pok.,https://adresowo.pl/o/mieszkanie-warszawa-remb...
26,Warszawa Mokotów - ul. Stefana Batorego,919 000 zł,37 m²,2 pok.,https://adresowo.pl/o/mieszkanie-warszawa-moko...
27,Warszawa Mokotów - ul. Bokserska,760 000 zł,35 m²,2 pok.,https://adresowo.pl/o/mieszkanie-warszawa-moko...
28,Warszawa Rembertów - ul. Stanisława Fiszera,zapytaj,37 m²,2 pok.,https://adresowo.pl/o/mieszkanie-warszawa-remb...
30,Warszawa Ochota - ul. Wiślicka,848 000 zł,49 m²,3 pok.,https://adresowo.pl/o/mieszkanie-warszawa-ocho...


In [31]:
import pandas as pd
import requests
import math
import time

# --- FUNKCJE POMOCNICZE ---

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)
    dlon, dlat = lon2_rad - lon1_rad, lat2_rad - lat1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    return R * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))

def clean_address_for_geo(text):
    if ' - ' in text:
        parts = text.split(' - ')
        street = parts[-1].strip()
        city = parts[0].split()[0].strip()
        
        # Usuń "ul. " na początku ulicy, jeśli występuje
        if street.startswith('ul. '):
            street = street[4:].strip()
        
        # LOGIKA: Jeśli nie ma cyfry w nazwie ulicy, dodajemy ' 1'
        if not any(char.isdigit() for char in street):
            street = f"{street} 1"
            
        return f"{street}, {city}"
    return text

def geocode_address(address):
    clean_addr = clean_address_for_geo(address)
    url = f"https://nominatim.openstreetmap.org/search?q={requests.utils.quote(clean_addr)}&format=json&limit=1"
    headers = {'User-Agent': 'Property_Analysis_Bot_v1'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200 and response.json():
            data = response.json()[0]
            return float(data['lat']), float(data['lon'])
    except Exception:
        pass
    return None, None

def get_nearest_distances(lat, lon, radius=2000):
    headers = {
        'User-Agent': 'Property_Analysis_Bot_v1',
        'Accept': 'application/json'
    }
    overpass_url = "https://overpass-api.de/api/interpreter"
    overpass_query = f"""
    [out:json][timeout:25];
    (
      nwr["highway"="bus_stop"](around:{radius},{lat},{lon});
      nwr["amenity"="school"](around:{radius},{lat},{lon});
    );
    out center;
    """
    try:
        response = requests.post(overpass_url, data=overpass_query, headers=headers, timeout=30)
        print(f"Overpass status: {response.status_code}")
        if response.status_code != 200:
            print(f"Overpass error: {response.text[:200]}")
            return None, None
            
        data = response.json()
        elements = data.get('elements', [])
        print(f"Liczba znalezionych elementów: {len(elements)}")
        
        min_bus, min_school = float('inf'), float('inf')
        bus_count, school_count = 0, 0
        
        for element in elements:
            e_lat = element.get('lat') or element.get('center', {}).get('lat')
            e_lon = element.get('lon') or element.get('center', {}).get('lon')
            if not e_lat or not e_lon: continue
            
            dist = calculate_distance(lat, lon, e_lat, e_lon)
            tags = element.get('tags', {})
            
            if tags.get('highway') == 'bus_stop':
                min_bus = min(min_bus, dist)
                bus_count += 1
            elif tags.get('amenity') == 'school':
                min_school = min(min_school, dist)
                school_count += 1
        
        print(f"Znaleziono {bus_count} przystanków, {school_count} szkół")
        
        return (round(min_bus, 3) if min_bus != float('inf') else None, 
                round(min_school, 3) if min_school != float('inf') else None)
    except Exception as e:
        print(f"Błąd w get_nearest_distances: {e}")
        return None, None

# --- GŁÓWNY PROCES ---

processed_results = []

print("Rozpoczynam geokodowanie (z poprawką numeru budynku) i analizę odległości...")

# head(25) aby sprawdzić więcej wyników
for index, row in df.head(25).iterrows():
    title = row['Tytuł']
    print(f"[{index+1}] Przetwarzam: {clean_address_for_geo(title)}")
    
    lat, lon = geocode_address(title)
    time.sleep(1.2) # Respektujemy limit 1 zapytania na sekundę
    
    if lat and lon:
        bus, school = get_nearest_distances(lat, lon)
        processed_results.append({
            'Tytuł': title,
            'Cena': row['Cena'],
            'Metraż': row['Metraż'],
            'Pokoje': row['Pokoje'],
            'Lat': lat,
            'Lon': lon,
            'Bus_km': bus,
            'Szkola_km': school,
            'Link': row['Link']
        })
    else:
        print(f"  ! Lokalizacja nadal nieznaleziona dla: {title}")

final_df = pd.DataFrame(processed_results)
final_df[['Bus_km', 'Szkola_km']] = final_df[['Bus_km', 'Szkola_km']].fillna(0)

print("\nGotowe! Statystyki:")
print(f"Udało się zgeokodować {len(final_df)} z {len(df.head(25))} ofert.")
print(final_df.head())

Rozpoczynam geokodowanie (z poprawką numeru budynku) i analizę odległości...
[3] Przetwarzam: Limanowskiego 11, Warszawa
Overpass status: 200
Liczba znalezionych elementów: 158
Znaleziono 129 przystanków, 29 szkół
[6] Przetwarzam: Wita Stwosza 1, Warszawa
Overpass status: 200
Liczba znalezionych elementów: 201
Znaleziono 167 przystanków, 34 szkół
[20] Przetwarzam: al. gen. Władysława Sikorskiego 1, Warszawa
Overpass status: 200
Liczba znalezionych elementów: 182
Znaleziono 145 przystanków, 37 szkół
[21] Przetwarzam: Jana Maklakiewicza 1, Warszawa
Overpass status: 200
Liczba znalezionych elementów: 216
Znaleziono 172 przystanków, 44 szkół
[22] Przetwarzam: Bitwy Warszawskiej 1920 r., Warszawa
Overpass status: 200
Liczba znalezionych elementów: 206
Znaleziono 159 przystanków, 47 szkół
[23] Przetwarzam: Rembertów 1, Warszawa
Overpass status: 504
Overpass error: <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    "http://www.w3.org/TR/xhtml1/

In [32]:
final_df

,Tytuł,Cena,Metraż,Pokoje,Lat,Lon,Bus_km,Szkola_km,Link
0,Warszawa Mokotów - Limanowskiego 11,1 420 000 zł,99 m²,5 pok.,52.190388,21.061693,0.127,0.066,https://adresowo.pl/o/mieszkanie-warszawa-moko...
1,Warszawa Mokotów - ul. Wita Stwosza,851 400 zł,38 m²,1 pok.,52.173985,21.018334,0.065,0.380,https://adresowo.pl/o/mieszkanie-warszawa-moko...
2,Warszawa Mokotów - al. gen. Władysława Sikorsk...,1 790 000 zł,67 m²,3 pok.,52.176091,21.044097,0.046,0.357,https://adresowo.pl/o/mieszkanie-warszawa-moko...
3,Warszawa Mokotów - ul. Jana Maklakiewicza,759 000 zł,48 m²,3 pok.,52.188969,20.992686,0.024,0.238,https://adresowo.pl/o/mieszkanie-warszawa-moko...
4,Warszawa Ochota - ul. Bitwy Warszawskiej 1920 r.,850 000 zł,34 m²,2 pok.,52.212622,20.973121,0.077,0.264,https://adresowo.pl/o/mieszkanie-warszawa-ocho...
5,Warszawa - Rembertów,1 649 000 zł,89 m²,4 pok.,52.270725,21.180787,0.000,0.000,https://adresowo.pl/o/mieszkanie-warszawa-remb...
6,Warszawa Mokotów - ul. Stefana Batorego,919 000 zł,37 m²,2 pok.,52.212046,21.015087,0.112,0.190,https://adresowo.pl/o/mieszkanie-warszawa-moko...
7,Warszawa Mokotów - ul. Bokserska,760 000 zł,35 m²,2 pok.,52.167173,21.007329,0.307,0.156,https://adresowo.pl/o/mieszkanie-warszawa-moko...
8,Warszawa Rembertów - ul. Stanisława Fiszera,zapytaj,37 m²,2 pok.,52.257857,21.167028,0.016,0.040,https://adresowo.pl/o/mieszkanie-warszawa-remb...
9,Warszawa Ochota - ul. Wiślicka,848 000 zł,49 m²,3 pok.,52.202467,20.977587,0.130,0.278,https://adresowo.pl/o/mieszkanie-warszawa-ocho...
